In [1]:
import pandas as pd
import numpy as np
import os
import warnings
from tqdm import tqdm
from PIL import Image
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import ModelCheckpoint, LearningRateScheduler
from tensorflow.keras import layers
warnings.filterwarnings('ignore')

2026-01-15 11:42:29.844683: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1768477350.024148      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768477350.074134      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1768477350.516929      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768477350.516972      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768477350.516975      23 computation_placer.cc:177] computation placer alr

In [2]:
BASE_DIRS = [r'/kaggle/input/utkface-new/UTKFace']
image_paths = []
age_labels = []
gender_labels = []

In [3]:
print("Reading file paths...")
for base_dir in BASE_DIRS:
    for filename in tqdm(os.listdir(base_dir)):
        temp = filename.split('_')
        # Basic error checking for filenames
        if len(temp) > 2 and temp[0].isdigit() and temp[1].isdigit():
            age = int(temp[0])
            gender = int(temp[1])
            image_path = os.path.join(base_dir, filename)
            age_labels.append(age)
            gender_labels.append(gender)
            image_paths.append(image_path)

Reading file paths...


100%|██████████| 23708/23708 [00:00<00:00, 544498.86it/s]


In [4]:
df = pd.DataFrame()
df['image'], df['age'], df['gender'] = image_paths, age_labels, gender_labels

In [5]:
def extract_feature(images):
    features = []
    print("Loading and resizing images to 128x128 RGB...")
    for image in tqdm(images):
        try:
            # CHANGED: convert('RGB') instead of 'L'
            img = Image.open(image).convert('RGB')
            img = img.resize((128, 128))
            img = np.array(img, dtype=np.float32)/ 255.0
            features.append(img)
        except:
            pass
    return np.array(features, dtype=np.float32)

In [6]:
x = extract_feature(df['image'])

Loading and resizing images to 128x128 RGB...


100%|██████████| 23708/23708 [03:38<00:00, 108.27it/s]


In [7]:
gender_label = np.array(df['gender'])
age_label = np.array(df['age'])
labels = np.column_stack((gender_label, age_label))
indices = np.arange(len(x))
np.random.shuffle(indices)
x = x[indices]
labels = labels[indices]

In [8]:
input_shape = (128, 128, 3)
inputs = Input(input_shape)

a = layers.RandomFlip("horizontal")(inputs)
a = layers.RandomRotation(0.1)(a)
a = layers.RandomZoom(0.1)(a)

a = Conv2D(32, kernel_size=(3,3), activation='relu')(a)
a = BatchNormalization()(a)
a = MaxPooling2D(pool_size=(2,2))(a)

a = Conv2D(64, kernel_size=(3,3), activation='relu')(a)
a = MaxPooling2D(pool_size=(2,2))(a)

a = Conv2D(128, kernel_size=(3,3), activation='relu')(a)
a = MaxPooling2D(pool_size=(2,2))(a)

a = Conv2D(256, kernel_size=(3,3), activation='relu')(a)
a = MaxPooling2D(pool_size=(2,2))(a)

a = Conv2D(512, kernel_size=(3,3), activation='relu')(a)
a = MaxPooling2D(pool_size=(2,2))(a)

a = Flatten()(a)

# Gender Branch
dense1 = Dense(256, activation='relu')(a)
dropout1 = Dropout(0.3)(dense1)
output1 = Dense(1, activation='sigmoid', name='gender_out')(dropout1)

# Age Branch
dense2 = Dense(256, activation='relu')(a)
dropout2 = Dropout(0.3)(dense2)
output2 = Dense(1, activation='relu', name='age_out')(dropout2)

model = Model(inputs=inputs, outputs=[output1, output2])
model.summary()

I0000 00:00:1768477584.808262      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13942 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1768477584.812124      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13942 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 128, 128,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ random_flip         │ (None, 128, 128,  │          0 │ input_layer[0][0] │
│ (RandomFlip)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ random_rotation     │ (None, 128, 128,  │          0 │ random_flip[0][0] │
│ (RandomRotation)    │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ random_zoom         │ (None, 128, 128,  │          0 │ random_rotation[… │
│ (RandomZoom)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 126, 126,  │        896 │ random_zoom[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 126, 126,  │        128 │ conv2d[0][0]      │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 63, 63,    │          0 │ batch_normalizat… │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 61, 61,    │     18,496 │ max_pooling2d[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 30, 30,    │          0 │ conv2d_1[0][0]    │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 28, 28,    │     73,856 │ max_pooling2d_1[… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_2     │ (None, 14, 14,    │          0 │ conv2d_2[0][0]    │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 12, 12,    │    295,168 │ max_pooling2d_2[… │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_3     │ (None, 6, 6, 256) │          0 │ conv2d_3[0][0]    │
│ (MaxPooling2D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 4, 4, 512) │  1,180,160 │ max_pooling2d_3[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_4     │ (None, 2, 2, 512) │          0 │ conv2d_4[0][0]    │
│ (MaxPooling2D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 2048)      │          0 │ max_pooling2d_4[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 256)       │    524,544 │ flatten[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 256)       │    524,544 │ flatten[0][0]   

 Total params: 2,618,306 (9.99 MB)

 Trainable params: 2,618,242 (9.99 MB)

 Non-trainable params: 64 (256.00 B)

In [9]:
x_train, x_temp, y_train, y_temp = train_test_split(x, labels, test_size=0.2, random_state=42)
x_test, x_val, y_test, y_val = train_test_split(x_temp, y_temp, test_size=0.5, random_state=42)
train_gender, train_age = y_train[:,0], y_train[:,1]
val_gender, val_age = y_val[:,0], y_val[:,1]
test_gender, test_age = y_test[:,0], y_test[:,1]

In [10]:
model_path = './best_224_rgb_model.keras'
checkpointer = ModelCheckpoint(
    filepath=model_path,
    verbose=1,
    save_best_only=True,
    monitor='val_loss',
    mode='min'
)

model.compile(
    loss={'gender_out': 'binary_crossentropy', 'age_out': 'mae'},
    optimizer='adam',
    metrics={'gender_out': 'accuracy', 'age_out': 'mae'}
)

annealer = LearningRateScheduler(lambda x: 1e-3 * 0.95 ** x)

print("Starting Training...")
history = model.fit(
    x=x_train,
    y=[train_gender, train_age],
    batch_size=32,
    epochs=50,
    validation_data=(x_val, [val_gender, val_age]),
    callbacks=[annealer, checkpointer]
)

Starting Training...
Epoch 1/50


I0000 00:00:1768477600.882900      68 cuda_dnn.cc:529] Loaded cuDNN version 91002


593/593 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - age_out_loss: 15.8770 - age_out_mae: 15.8770 - gender_out_accuracy: 0.5539 - gender_out_loss: 0.7235 - loss: 16.6005
Epoch 1: val_loss improved from inf to 23.15043, saving model to ./best_224_rgb_model.keras
593/593 ━━━━━━━━━━━━━━━━━━━━ 31s 40ms/step - age_out_loss: 15.8742 - age_out_mae: 15.8742 - gender_out_accuracy: 0.5540 - gender_out_loss: 0.7234 - loss: 16.5976 - val_age_out_loss: 22.3580 - val_age_out_mae: 22.5263 - val_gender_out_accuracy: 0.6647 - val_gender_out_loss: 0.6277 - val_loss: 23.1504 - learning_rate: 0.0010
Epoch 2/50
593/593 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - age_out_loss: 11.3872 - age_out_mae: 11.3872 - gender_out_accuracy: 0.6923 - gender_out_loss: 0.5933 - loss: 11.9806
Epoch 2: val_loss improved from 23.15043 to 10.16485, saving model to ./best_224_rgb_model.keras
593/593 ━━━━━━━━━━━━━━━━━━━━ 22s 37ms/step - age_out_loss: 11.3866 - age_out_mae: 11.3866 - gender_out_accuracy: 0.6923 - gender_out_loss: 0.5933 - loss: